# Prepare Western District access layers

Use GeoPandas to normalize Hong Kong open spatial datasets, work in a metric CRS for buffers/clipping, and export WGS84 GeoJSON for the web map. Download current layers from the [Hong Kong GeoData Store](https://portal.csdi.gov.hk/) or DATA.GOV.HK. The site's checked-in `static/data/access-points.geojson` is illustrative only.

Suggested inputs: CSDI / GeoInfo public open-space polygons and LCSD Outdoor Fitness Equipment point layer. Keep the original source, download date, licence, and field mapping in the notes below.

In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd

ROOT = Path('..').resolve()
SOURCE_DIR = ROOT / 'data' / 'source'
OUTPUT = ROOT / 'static' / 'data'
OUTPUT.mkdir(parents=True, exist_ok=True)

# Set these paths to downloaded CSDI / GeoInfo GeoJSON, GML, or Shapefile layers.
GREEN_FILE = SOURCE_DIR / 'public_green_space.geojson'
GYM_FILE = SOURCE_DIR / 'outdoor_fitness_equipment.geojson'

# Hong Kong 1980 Grid is suitable for local metric buffers and distance operations.
METRIC_CRS = 'EPSG:2326'
WEB_CRS = 'EPSG:4326'
ORIGIN_LON_LAT = (114.1285, 22.2833)  # approximate placeholder; survey/verify entrance

## Read and inspect sources

Check field names and coordinate systems before proceeding. Public-space polygons should remain polygons so the clip step identifies the portion intersecting the walking buffer; fitness equipment is commonly represented as point locations.

In [ ]:
def read_layer(path: Path) -> gpd.GeoDataFrame:
    if not path.exists():
        raise FileNotFoundError(f'Missing downloaded source layer: {path}')
    layer = gpd.read_file(path)
    if layer.crs is None:
        raise ValueError(f'{path.name} has no CRS; assign its documented source CRS before reprojection')
    return layer

green = read_layer(GREEN_FILE)
fitness = read_layer(GYM_FILE)
print('Green CRS / rows:', green.crs, len(green))
print('Fitness CRS / rows:', fitness.crs, len(fitness))
display(green.head(3))
display(fitness.head(3))

## Reproject, filter to the study area, and clip

The example creates a 2 km straight-line *study extent* to keep output focused around the origin. This is only a data-preparation extent, not the final walking isochrone. Actual access inclusion is based on route distance and sampled grade in the app.

In [ ]:
origin_web = gpd.GeoSeries([gpd.points_from_xy([ORIGIN_LON_LAT[0]], [ORIGIN_LON_LAT[1]])[0]], crs=WEB_CRS)
origin_metric = origin_web.to_crs(METRIC_CRS).iloc[0]
study_area = gpd.GeoSeries([origin_metric.buffer(2_000)], crs=METRIC_CRS).iloc[0]

green_m = green.to_crs(METRIC_CRS)
fitness_m = fitness.to_crs(METRIC_CRS)
green_clip = green_m.loc[green_m.intersects(study_area)].copy()
green_clip['geometry'] = green_clip.geometry.intersection(study_area)
fitness_clip = fitness_m.loc[fitness_m.within(study_area)].copy()
print(f'{len(green_clip)} green polygons and {len(fitness_clip)} fitness points in 2 km prep extent')

## Normalize features and export web layers

Update `name_column` and optional identifier fields to match the source layer's documented schema. Do not discard source identifiers or metadata needed to trace records back to the publisher.

In [ ]:
def normalize(gdf: gpd.GeoDataFrame, kind: str, name_column: str) -> gpd.GeoDataFrame:
    out = gdf.to_crs(WEB_CRS).copy()
    out['kind'] = kind
    out['name'] = out[name_column].fillna(kind.title()) if name_column in out else kind.title()
    out['source'] = 'Hong Kong GeoData Store / source layer metadata'
    return out

# Replace these field names after inspecting the downloaded layers.
green_web = normalize(green_clip, 'green', 'name_en')
fitness_web = normalize(fitness_clip, 'gym', 'facility_name_en')
combined = gpd.GeoDataFrame(pd.concat([green_web, fitness_web], ignore_index=True), crs=WEB_CRS)
combined.to_file(OUTPUT / 'access-points.geojson', driver='GeoJSON', index=False)
print('Wrote', OUTPUT / 'access-points.geojson', 'features:', len(combined))

## Notes for interpreting the map

A Euclidean GeoPandas buffer is useful for clipping source layers but does not equal an on-foot catchment. The app obtains pedestrian route distances and samples Google elevation along each route when Google Maps Platform is configured. Its slope test estimates the steepest absolute segment angle between samples. Elevation raster resolution and sampling can miss short ramps or steps, and a routed path may omit accessibility details such as kerbs, stairs, crossing signals, surface quality, closures, or entrances. Field verification and a pedestrian network with step-free attributes are needed before product or policy decisions.

For reproducibility, record source URLs, retrieval date, licence, source CRS, name/ID field mappings, and any filters applied when preparing each layer.